# TE02_029 - Digital Twin Update Frequency Validation

This notebook validates the KPI: **Digital twin update frequency < 5 seconds**.

Validation path:

`/joint_states` from the real machine -> Unity visual model update -> `/joints_sim` and `/digital_twin/joint_state_sync_status` publication.

The KPI is evaluated from Unity's `/digital_twin/joint_state_sync_status` topic. Each message is JSON inside `std_msgs/String` and contains the latency between the real `/joint_states` source timestamp and the time after Unity applies the update.

## Acceptance Criteria

A digital twin update is valid when Unity applies a real `/joint_states` update and reports an update latency below `5000 ms`.

The KPI passes when:

- at least `MIN_SYNC_SAMPLES` sync-status samples are collected
- `max(latency_ms) < 5000 ms`
- `p95(latency_ms) < 5000 ms`
- all reported latencies are non-negative

Supporting evidence is also collected from `/joints_sim` to confirm that Unity publishes the simulated joint vector.

In [1]:
from pathlib import Path
import csv
import json
import math
import statistics
import time
import os
# ROS imports are intentionally kept in a setup cell so the rest of the notebook
# can still be inspected on machines without a sourced ROS environment.
import rclpy
from rclpy.node import Node
from std_msgs.msg import String
from sensor_msgs.msg import JointState

KPI_DIR = Path(str(os.environ.get('HOME')) + '/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_029')
KPI_DIR.mkdir(parents=True, exist_ok=True)

HOST_NAME = os.environ.get('HOSTNAME')

SYNC_STATUS_TOPIC = '/digital_twin/joint_state_sync_status'
JOINTS_SIM_TOPIC = '/joints_sim'
REAL_JOINT_STATES_TOPIC = '/joint_states'

COLLECTION_DURATION_S = 120.0
THRESHOLD_MS = 5000.0
MIN_SYNC_SAMPLES = 30

SYNC_SAMPLES_CSV = KPI_DIR / 'te02_029_sync_samples.csv'
JOINTS_SIM_CSV = KPI_DIR / 'te02_029_joints_sim_samples.csv'
SUMMARY_CSV = KPI_DIR / 'te02_029_summary.csv'

CLOCK_SYNC_NOTES = {
    'simulation_machine': str(HOST_NAME),
    'simulation_rms_offset_ms': 1.709817,
    'simulation_root_dispersion_ms': 1.290345,
    'simulation_leap_status': 'Normal',
    'real_machine': 'mitac-telehandler-0',
    'real_machine_rms_offset_ms': 4.701989,
    'real_machine_root_dispersion_ms': 4.251608,
    'real_machine_leap_status': 'Normal',
}

print(f'Output directory: {KPI_DIR}')

Output directory: /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_029


In [2]:
def percentile(values, pct):
    if not values:
        return math.nan
    ordered = sorted(values)
    if len(ordered) == 1:
        return ordered[0]
    rank = (pct / 100.0) * (len(ordered) - 1)
    lower = math.floor(rank)
    upper = math.ceil(rank)
    if lower == upper:
        return ordered[int(rank)]
    fraction = rank - lower
    return ordered[lower] * (1.0 - fraction) + ordered[upper] * fraction

def stamp_to_sec(stamp):
    return float(stamp.sec) + float(stamp.nanosec) * 1e-9

def write_csv(path, rows, fieldnames):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def now_ns():
    return time.time_ns()

In [3]:
class TE02029Collector(Node):
    def __init__(self):
        super().__init__('te02_029_digital_twin_sync_collector')
        self.sync_samples = []
        self.joints_sim_samples = []
        self.real_joint_state_samples = []
        self.create_subscription(String, SYNC_STATUS_TOPIC, self.on_sync_status, 50)
        self.create_subscription(JointState, JOINTS_SIM_TOPIC, self.on_joints_sim, 50)
        self.create_subscription(JointState, REAL_JOINT_STATES_TOPIC, self.on_real_joint_states, 50)

    def on_sync_status(self, msg):
        receive_time_ns = now_ns()
        try:
            payload = json.loads(msg.data)
        except json.JSONDecodeError as exc:
            self.sync_samples.append({
                'receive_time_ns': receive_time_ns,
                'parse_ok': False,
                'error': str(exc),
                'source_topic': '',
                'source_stamp_sec': '',
                'unity_update_time_sec': '',
                'latency_ms': '',
                'joint_count': '',
                'raw_data': msg.data,
            })
            return

        self.sync_samples.append({
            'receive_time_ns': receive_time_ns,
            'parse_ok': True,
            'error': '',
            'source_topic': payload.get('source_topic', ''),
            'source_stamp_sec': payload.get('source_stamp_sec', ''),
            'unity_update_time_sec': payload.get('unity_update_time_sec', ''),
            'latency_ms': payload.get('latency_ms', ''),
            'joint_count': payload.get('joint_count', ''),
            'raw_data': msg.data,
        })

    def on_joints_sim(self, msg):
        receive_time_ns = now_ns()
        self.joints_sim_samples.append({
            'receive_time_ns': receive_time_ns,
            'header_stamp_sec': stamp_to_sec(msg.header.stamp),
            'frame_id': msg.header.frame_id,
            'joint_count': len(msg.name),
            'joint_names': ';'.join(msg.name),
            'positions': ';'.join(str(v) for v in msg.position),
        })

    def on_real_joint_states(self, msg):
        self.real_joint_state_samples.append({
            'receive_time_ns': now_ns(),
            'header_stamp_sec': stamp_to_sec(msg.header.stamp),
            'joint_count': len(msg.name),
        })

In [4]:
if not rclpy.ok():
    rclpy.init()

collector = TE02029Collector()
start = time.monotonic()
print(f'Collecting TE02_029 evidence for {COLLECTION_DURATION_S:.1f} s...')

try:
    while time.monotonic() - start < COLLECTION_DURATION_S:
        rclpy.spin_once(collector, timeout_sec=0.1)
finally:
    elapsed = time.monotonic() - start

print(f'Collected for {elapsed:.2f} s')
print(f'sync-status samples: {len(collector.sync_samples)}')
print(f'/joints_sim samples: {len(collector.joints_sim_samples)}')
print(f'/joint_states samples: {len(collector.real_joint_state_samples)}')

Collected for 120.05 s
sync-status samples: 1197
/joints_sim samples: 2367
/joint_states samples: 1194


In [5]:
sync_fieldnames = [
    'receive_time_ns', 'parse_ok', 'error', 'source_topic', 'source_stamp_sec',
    'unity_update_time_sec', 'latency_ms', 'joint_count', 'raw_data'
]
joints_sim_fieldnames = [
    'receive_time_ns', 'header_stamp_sec', 'frame_id', 'joint_count', 'joint_names', 'positions'
]

write_csv(SYNC_SAMPLES_CSV, collector.sync_samples, sync_fieldnames)
write_csv(JOINTS_SIM_CSV, collector.joints_sim_samples, joints_sim_fieldnames)

valid_sync = []
invalid_latency_count = 0
for row in collector.sync_samples:
    if not row.get('parse_ok'):
        continue
    try:
        latency = float(row['latency_ms'])
    except (TypeError, ValueError):
        continue
    if latency < 0.0:
        invalid_latency_count += 1
    valid_sync.append(latency)

joints_sim_receive_times_s = [row['receive_time_ns'] * 1e-9 for row in collector.joints_sim_samples]
joints_sim_gaps_s = [b - a for a, b in zip(joints_sim_receive_times_s, joints_sim_receive_times_s[1:])]
real_receive_times_s = [row['receive_time_ns'] * 1e-9 for row in collector.real_joint_state_samples]
real_gaps_s = [b - a for a, b in zip(real_receive_times_s, real_receive_times_s[1:])]

summary_rows = []
def add(metric, value, status='INFO', details=''):
    summary_rows.append({'metric': metric, 'value': value, 'status': status, 'details': details})

sync_sample_count = len(valid_sync)
parse_error_count = sum(1 for row in collector.sync_samples if not row.get('parse_ok'))
samples_over_threshold = sum(1 for latency in valid_sync if latency >= THRESHOLD_MS)
success_rate = ((sync_sample_count - samples_over_threshold) / sync_sample_count * 100.0) if sync_sample_count else 0.0

latency_mean = statistics.mean(valid_sync) if valid_sync else math.nan
latency_median = statistics.median(valid_sync) if valid_sync else math.nan
latency_p95 = percentile(valid_sync, 95)
latency_max = max(valid_sync) if valid_sync else math.nan
latency_min = min(valid_sync) if valid_sync else math.nan

status_sample_count = 'PASS' if sync_sample_count >= MIN_SYNC_SAMPLES else 'FAIL'
status_non_negative = 'PASS' if invalid_latency_count == 0 and sync_sample_count > 0 else 'FAIL'
status_max_latency = 'PASS' if sync_sample_count > 0 and latency_max < THRESHOLD_MS else 'FAIL'
status_p95_latency = 'PASS' if sync_sample_count > 0 and latency_p95 < THRESHOLD_MS else 'FAIL'
status_joints_sim = 'PASS' if len(collector.joints_sim_samples) > 0 else 'FAIL'
overall = 'PASS' if all(s == 'PASS' for s in [status_sample_count, status_non_negative, status_max_latency, status_p95_latency, status_joints_sim]) else 'FAIL'

add('clock_sync_simulation_machine', CLOCK_SYNC_NOTES['simulation_machine'], 'INFO', f"rms_offset_ms={CLOCK_SYNC_NOTES['simulation_rms_offset_ms']}; root_dispersion_ms={CLOCK_SYNC_NOTES['simulation_root_dispersion_ms']}; leap_status={CLOCK_SYNC_NOTES['simulation_leap_status']}")
add('clock_sync_real_machine', CLOCK_SYNC_NOTES['real_machine'], 'INFO', f"rms_offset_ms={CLOCK_SYNC_NOTES['real_machine_rms_offset_ms']}; root_dispersion_ms={CLOCK_SYNC_NOTES['real_machine_root_dispersion_ms']}; leap_status={CLOCK_SYNC_NOTES['real_machine_leap_status']}")
add('collection_duration_s', round(elapsed, 3), 'INFO')
add('threshold_ms', THRESHOLD_MS, 'INFO')
add('sync_status_topic', SYNC_STATUS_TOPIC, 'INFO')
add('joints_sim_topic', JOINTS_SIM_TOPIC, 'INFO')
add('real_joint_states_topic', REAL_JOINT_STATES_TOPIC, 'INFO')
add('sync_status_samples', sync_sample_count, status_sample_count, f'min_required={MIN_SYNC_SAMPLES}')
add('sync_status_parse_errors', parse_error_count, 'PASS' if parse_error_count == 0 else 'FAIL')
add('negative_latency_samples', invalid_latency_count, status_non_negative)
add('latency_min_ms', round(latency_min, 3) if valid_sync else '', 'INFO')
add('latency_mean_ms', round(latency_mean, 3) if valid_sync else '', 'INFO')
add('latency_median_ms', round(latency_median, 3) if valid_sync else '', 'INFO')
add('latency_p95_ms', round(latency_p95, 3) if valid_sync else '', status_p95_latency, f'threshold<{THRESHOLD_MS} ms')
add('latency_max_ms', round(latency_max, 3) if valid_sync else '', status_max_latency, f'threshold<{THRESHOLD_MS} ms')
add('samples_over_threshold', samples_over_threshold, 'PASS' if samples_over_threshold == 0 and sync_sample_count > 0 else 'FAIL')
add('success_rate_pct', round(success_rate, 3), 'PASS' if success_rate == 100.0 and sync_sample_count > 0 else 'FAIL')
add('joints_sim_samples', len(collector.joints_sim_samples), status_joints_sim)
add('joints_sim_max_gap_s', round(max(joints_sim_gaps_s), 6) if joints_sim_gaps_s else '', 'INFO')
add('real_joint_states_samples', len(collector.real_joint_state_samples), 'INFO')
add('real_joint_states_max_gap_s', round(max(real_gaps_s), 6) if real_gaps_s else '', 'INFO')
add('TE02_029_overall', overall, overall, f'{sync_sample_count} valid Unity sync latency samples; max_latency_ms={latency_max:.3f}' if valid_sync else 'no valid sync latency samples')

write_csv(SUMMARY_CSV, summary_rows, ['metric', 'value', 'status', 'details'])

print(f'Wrote {SYNC_SAMPLES_CSV}')
print(f'Wrote {JOINTS_SIM_CSV}')
print(f'Wrote {SUMMARY_CSV}')
print(f'TE02_029 overall: {overall}')

Wrote /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_029/te02_029_sync_samples.csv
Wrote /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_029/te02_029_joints_sim_samples.csv
Wrote /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_029/te02_029_summary.csv
TE02_029 overall: PASS


In [6]:
for row in summary_rows:
    print(f"{row['metric']}: {row['value']} [{row['status']}] {row['details']}")

clock_sync_simulation_machine: None [INFO] rms_offset_ms=1.709817; root_dispersion_ms=1.290345; leap_status=Normal
clock_sync_real_machine: mitac-telehandler-0 [INFO] rms_offset_ms=4.701989; root_dispersion_ms=4.251608; leap_status=Normal
collection_duration_s: 120.048 [INFO] 
threshold_ms: 5000.0 [INFO] 
sync_status_topic: /digital_twin/joint_state_sync_status [INFO] 
joints_sim_topic: /joints_sim [INFO] 
real_joint_states_topic: /joint_states [INFO] 
sync_status_samples: 1197 [PASS] min_required=30
sync_status_parse_errors: 0 [PASS] 
negative_latency_samples: 0 [PASS] 
latency_min_ms: 4.805 [INFO] 
latency_mean_ms: 29.483 [INFO] 
latency_median_ms: 16.994 [INFO] 
latency_p95_ms: 78.744 [PASS] threshold<5000.0 ms
latency_max_ms: 526.462 [PASS] threshold<5000.0 ms
samples_over_threshold: 0 [PASS] 
success_rate_pct: 100.0 [PASS] 
joints_sim_samples: 2367 [PASS] 
joints_sim_max_gap_s: 0.417492 [INFO] 
real_joint_states_samples: 1194 [INFO] 
real_joint_states_max_gap_s: 0.375075 [INFO] 
T

In [7]:
collector.destroy_node()
rclpy.shutdown()